После этого — план для следующего ноутбука (02_asymmetry_analysis.ipynb):

Вопрос 2 — "rockets and feathers": растут ли цены на заправке быстрее в ответ на рост Brent, чем падают в ответ на падение Brent. Та же логика regression, что уже сделал, но с одним структурным изменением:

Нужно разложить brent_pct (и его лаги) на два отдельных регрессора: "положительная часть" (когда Brent вырос, иначе 0) и "отрицательная часть" (когда Brent упал, иначе 0)

Прогнать ту же OLS-регрессию, но с этими раздельными переменными вместо единого brent_pct

Сравнить коэффициенты — если коэффициент при "росте" заметно больше по модулю, чем при "падении", это подтверждает asymmetry (feathers эффект — цены быстро растут, медленно падают)

In [1]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/weekly_pump_brent_eur_2019_2026.parquet")
df.head()

,week_date,petrol_price_pretax,diesel_price_pretax,brent_date,close_price,eur_usd_date,close_rate,brent_eur,brent_pct,pump_pct,eur_usd_pct,brent_pct_lag1,brent_pct_lag2,eur_usd_pct_lag1,eur_usd_pct_lag2
0,2019-01-21,464.65,524.56,2019-01-18,62.700001,2019-01-21,1.136557,55.166591,0.062892,-0.033811,-0.008251,0.028955,-0.038249,0.004355,0.009243
1,2019-01-28,464.65,557.08,2019-01-28,59.930000,2019-01-28,1.141305,52.510067,-0.044179,0.000000,0.004177,0.062892,0.028955,-0.008251,0.004355
2,2019-02-04,472.78,557.08,2019-02-04,62.509998,2019-02-04,1.145528,54.568729,0.043050,0.017497,0.003700,-0.044179,0.062892,0.004177,-0.008251
3,2019-02-11,489.04,557.08,2019-02-11,61.509998,2019-02-11,1.132426,54.317018,-0.015997,0.034392,-0.011437,0.043050,-0.044179,0.003700,0.004177
4,2019-02-18,471.96,557.08,2019-02-15,66.250000,2019-02-18,1.129803,58.638538,0.077061,-0.034926,-0.002316,-0.015997,0.043050,-0.011437,0.003700


In [2]:
df.columns

Index(['week_date', 'petrol_price_pretax', 'diesel_price_pretax', 'brent_date',
       'close_price', 'eur_usd_date', 'close_rate', 'brent_eur', 'brent_pct',
       'pump_pct', 'eur_usd_pct', 'brent_pct_lag1', 'brent_pct_lag2',
       'eur_usd_pct_lag1', 'eur_usd_pct_lag2'],
      dtype='str')

In [4]:
df['brent_pct_up'] = np.where(df['brent_pct'] > 0, df['brent_pct'], 0)
df['brent_pct_down'] = np.where(df['brent_pct'] < 0, df['brent_pct'], 0)

In [5]:
df['brent_pct_lag1_up'] = np.where(df['brent_pct_lag1'] > 0, df['brent_pct_lag1'], 0)
df['brent_pct_lag1_down'] = np.where(df['brent_pct_lag1'] < 0, df['brent_pct_lag1'], 0)

df['brent_pct_lag2_up'] = np.where(df['brent_pct_lag2'] > 0, df['brent_pct_lag2'], 0)
df['brent_pct_lag2_down'] = np.where(df['brent_pct_lag2'] < 0, df['brent_pct_lag2'], 0)


In [6]:
df[['brent_pct', 'brent_pct_up', 'brent_pct_down',
    'brent_pct_lag1', 'brent_pct_lag1_up', 'brent_pct_lag1_down',
    'brent_pct_lag2', 'brent_pct_lag2_up', 'brent_pct_lag2_down']].head(10)

,brent_pct,brent_pct_up,brent_pct_down,brent_pct_lag1,brent_pct_lag1_up,brent_pct_lag1_down,brent_pct_lag2,brent_pct_lag2_up,brent_pct_lag2_down
0,0.062892,0.062892,0.000000,0.028955,0.028955,0.000000,-0.038249,0.000000,-0.038249
1,-0.044179,0.000000,-0.044179,0.062892,0.062892,0.000000,0.028955,0.028955,0.000000
2,0.043050,0.043050,0.000000,-0.044179,0.000000,-0.044179,0.062892,0.062892,0.000000
3,-0.015997,0.000000,-0.015997,0.043050,0.043050,0.000000,-0.044179,0.000000,-0.044179
4,0.077061,0.077061,0.000000,-0.015997,0.000000,-0.015997,0.043050,0.043050,0.000000
5,-0.022491,0.000000,-0.022491,0.077061,0.077061,0.000000,-0.015997,0.000000,-0.015997
6,0.014052,0.014052,0.000000,-0.022491,0.000000,-0.022491,0.077061,0.077061,0.000000
7,0.013857,0.013857,0.000000,0.014052,0.014052,0.000000,-0.022491,0.000000,-0.022491
8,0.014419,0.014419,0.000000,0.013857,0.013857,0.000000,0.014052,0.014052,0.000000
9,-0.004886,0.000000,-0.004886,0.014419,0.014419,0.000000,0.013857,0.013857,0.000000


In [10]:
import statsmodels.api as sm

X = df[['brent_pct_up', 'brent_pct_down',
    'brent_pct_lag1_up', 'brent_pct_lag1_down',
    'brent_pct_lag2_up', 'brent_pct_lag2_down',
    'eur_usd_pct', 'eur_usd_pct_lag1', 'eur_usd_pct_lag2']]

X = sm.add_constant(X)

y = df['pump_pct']

model = sm.OLS(y, X).fit()
print(model.summary())



                            OLS Regression Results                            
Dep. Variable:               pump_pct   R-squared:                       0.262
Model:                            OLS   Adj. R-squared:                  0.244
Method:                 Least Squares   F-statistic:                     14.77
Date:                Wed, 12 Aug 2026   Prob (F-statistic):           1.53e-20
Time:                        16:50:17   Log-Likelihood:                 910.85
No. Observations:                 385   AIC:                            -1802.
Df Residuals:                     375   BIC:                            -1762.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   0.0044    

## Interpretation

Testing whether pump prices react asymmetrically to Brent increases vs
decreases ("rockets and feathers"):

- 1 week later: price increases pass through significantly (coef=0.156,
  p<0.001), price decreases don't (p=0.224)
- 2 weeks later: reversed — decreases pass through strongly (coef=0.329,
  p<0.001), increases don't (p=0.279)

Pattern: retail prices react to rising oil costs within a week, but the
response to falling costs is delayed to two weeks — consistent with
"rockets and feathers" behavior, where cost increases get passed on faster
than cost decreases.

EUR/USD lag2 is significant but with an unexpected negative sign, likely
noise rather than a real relationship — not a reliable driver here.

R²=0.26, slightly better fit than the symmetric pass-through model
(R²=0.21) — separating up/down movements captures real structure in how
retailers adjust prices.

In [11]:
df.to_parquet('../data/processed/weekly_pump_brent_eur_2019_2026_asymmetry.parquet', index=False)